#Sephora Products & Skincare Reviews — Data Cleaning Pipeline


## 1. Import Required Libraries


In [1]:
import pandas as pd
import numpy as np
import glob
import os
import kagglehub


## 2. Download Dataset from Kaggle


In [2]:
path = kagglehub.dataset_download("nadyinky/sephora-products-and-skincare-reviews")
print("Path to dataset files:", path)


100%|██████████| 147M/147M [00:00<00:00, 169MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/nadyinky/sephora-products-and-skincare-reviews/versions/2


## 3. Load Product Information Dataset


In [13]:
file_path = os.path.join(path, "product_info.csv")
product = pd.read_csv(file_path)
product.head()
product.info()
product.shape
product.describe()
product.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8494 entries, 0 to 8493
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_id          8494 non-null   object 
 1   product_name        8494 non-null   object 
 2   brand_id            8494 non-null   int64  
 3   brand_name          8494 non-null   object 
 4   loves_count         8494 non-null   int64  
 5   rating              8216 non-null   float64
 6   reviews             8216 non-null   float64
 7   size                6863 non-null   object 
 8   variation_type      7050 non-null   object 
 9   variation_value     6896 non-null   object 
 10  variation_desc      1250 non-null   object 
 11  ingredients         7549 non-null   object 
 12  price_usd           8494 non-null   float64
 13  value_price_usd     451 non-null    float64
 14  sale_price_usd      270 non-null    float64
 15  limited_edition     8494 non-null   int64  
 16  new   

,0
product_id,0
product_name,0
brand_id,0
brand_name,0
loves_count,0
rating,278
reviews,278
size,1631
variation_type,1444
variation_value,1598


## 4. Load and Combine Customer Reviews Datasets


In [14]:
search_pattern = os.path.join(path, "reviews_*.csv")
review_files = sorted(glob.glob(search_pattern))

review_list = []
for file in review_files:
    temp = pd.read_csv(file, low_memory=False)
    temp["source_file"] = os.path.basename(file)
    review_list.append(temp)

reviews = pd.concat(review_list, ignore_index=True, sort=False)


## 5. Handle Missing Values


In [15]:
for col in ["brand_name", "primary_category",
            "secondary_category", "tertiary_category"]:
    if col in product.columns:
        product[col] = product[col].fillna("Unknown")


## 6. Remove Duplicate Records


In [16]:
product = product.drop_duplicates()
reviews = reviews.drop_duplicates()

print("Products after duplicate removal:", product.shape[0])
print("Reviews after duplicate removal:", reviews.shape[0])


Products after duplicate removal: 8494
Reviews after duplicate removal: 1094411


## 7. Convert Numeric Columns


In [17]:
for col in ["price_usd", "rating", "reviews",
            "loves_count", "child_min_price", "child_max_price"]:
    if col in product.columns:
        product[col] = pd.to_numeric(product[col], errors="coerce")

if "rating" in reviews.columns:
    reviews["rating"] = pd.to_numeric(
        reviews["rating"], errors="coerce"
    )


## 8. Detect and Flag Price Outliers (IQR Method)


In [18]:
q1 = product["price_usd"].quantile(0.25)
q3 = product["price_usd"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

product["price_outlier"] = (
    (product["price_usd"] < lower) |
    (product["price_usd"] > upper)
)


## 9. Clean and Trim Text Columns


In [19]:
for col in ["product_name", "brand_name",
            "primary_category", "secondary_category",
            "tertiary_category"]:
    if col in product.columns:
        product[col] = product[col].astype("string").str.strip()


## 10. Convert Submission Date to Datetime Format


In [20]:
date_col = None
for col in ["submission_time", "date"]:
    if col in reviews.columns:
        date_col = col
        break

if date_col:
    reviews[date_col] = pd.to_datetime(
        reviews[date_col], errors="coerce"
    )


## 11. Merge Products and Reviews Datasets


In [21]:
product_unique = product.drop_duplicates(
    subset=["product_id"]
)

merged = reviews.merge(
    product_unique,
    on="product_id",
    how="left",
    suffixes=("_review", "_product")
)


## 12. Save Cleaned Datasets to CSV


In [22]:
product.to_csv("Sephora_Products_Cleaned.csv", index=False)
reviews.to_csv("Sephora_Reviews_Cleaned.csv", index=False)
merged.to_csv("Sephora_Merged_Cleaned.csv", index=False)

print("Cleaned files saved successfully!")


Cleaned files saved successfully!
